In [2]:
import numpy as np
import pandas as pd


In [3]:
df = pd.read_csv('Titanic_Dataset.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.drop(columns=['PassengerId', 'Name',  'Ticket','Cabin'], inplace = True)

In [6]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [7]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['Survived'], axis=1), df['Survived'], test_size=0.2, random_state=0)

In [10]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((712, 7), (179, 7), (712,), (179,))

In [52]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,3,female,NaN,0,2,15.2458,C
439,2,male,31.0,0,0,10.5000,S
817,2,male,31.0,1,1,37.0042,C
378,3,male,20.0,0,0,4.0125,C
491,3,male,21.0,0,0,7.2500,S


In [53]:
X_train.shape

(712, 7)

In [54]:
X_train['Age'].shape

(712,)

In [55]:
si_age = SimpleImputer(strategy='mean')
si_embarked = SimpleImputer(strategy='most_frequent')

In [56]:
X_train_age = si_age.fit_transform(X_train[['Age']])

X_test_age = si_age.transform(X_test[['Age']])

In [57]:
X_train_age.shape

(712, 1)

In [58]:
X_test_age.shape

(179, 1)

In [59]:
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

In [60]:
# transformer = ColumnTransformer([
#     ('tnf1',SimpleImputer(), ['Age'] ),
#     ('tnf2',OneHotEncoder(drop='first', sparse=False), [['Sex', 'Embarked']] )
    
# ],remainder='passthrough')

In [61]:
# one hot encoding on sex and embarked column
sex_ohe = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
embarked_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_sex = sex_ohe.fit_transform(X_train[['Sex']])
X_train_embarked = embarked_ohe.fit_transform(X_train_embarked)

# with test data 
X_test_sex  = sex_ohe.transform(X_test[['Sex']])
X_test_embarked = embarked_ohe.transform(X_test_embarked)


In [62]:
X_train_embarked.shape

(712, 3)

In [63]:
X_train.head(4)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,3,female,NaN,0,2,15.2458,C
439,2,male,31.0,0,0,10.5000,S
817,2,male,31.0,1,1,37.0042,C
378,3,male,20.0,0,0,4.0125,C


In [64]:
X_train_remaining_cols = X_train.drop(columns=['Sex', 'Age', 'Embarked'])
X_test_rem_cols = X_test.drop(columns=['Sex','Age', 'Embarked'])

In [65]:
X_train_remaining_cols.shape

(712, 4)

In [66]:
X_train_transformed = np.concatenate((X_train_remaining_cols, X_train_age,X_train_sex, X_train_embarked),axis=1)
X_test_transformed = np.concatenate((X_test_rem_cols, X_test_age,X_test_sex, X_test_embarked),axis=1)

In [67]:
X_train_transformed


array([[3., 0., 2., ..., 1., 0., 0.],
       [2., 0., 0., ..., 0., 0., 1.],
       [2., 1., 1., ..., 1., 0., 0.],
       ...,
       [3., 0., 0., ..., 0., 1., 0.],
       [3., 1., 0., ..., 0., 0., 1.],
       [2., 1., 1., ..., 0., 0., 1.]], shape=(712, 10))

<p> Until now we have done the data preprocessing that includes column transformation and then we concatenated </p>

<h1> Decision Trees 

In [68]:
from sklearn.tree import DecisionTreeClassifier

In [85]:
classifier = DecisionTreeClassifier()
classifier.fit(X_train_transformed, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [86]:
y_prediction = classifier.predict(X_train_transformed)

In [87]:
y_prediction.shape

(712,)

In [88]:
y_prediction = classifier.predict(X_test_transformed)

In [90]:
from sklearn.metrics import accuracy_score


In [93]:
accuracy = accuracy_score( y_test, y_prediction)

In [81]:
accuracy

0.9817415730337079

In [ ]:
accuracy = accuracy_score( y_test, y_prediction)

In [84]:
accuracy

0.776536312849162

In [94]:
import pickle

In [97]:
pickle.dump(sex_ohe, open('models/sex_ohe.pkl', 'wb'))
pickle.dump(embarked_ohe, open('models/embarked_ohe.pkl', 'wb'))
pickle.dump(classifier, open('models/classifier.pkl', 'wb'))